# SP-04 · Breakout Policy — `04_breakout_policy.ipynb`

Fourth specialised model of the **Personalized AI Trading Agent**. Produces `breakout_policy.pt` plus companion artifacts for the Meta-RL layer.

> **Delivered un-executed:** this notebook contains code only. Every number, table and plot is generated when *you* run it on the data you download.

> **Research and paper trading only.** Nothing here is investment advice. Historical backtests are not evidence of future profitability. All numbers are computed at run time from the data you download. None are pre-filled.

## What the model does
A GRU sequence classifier reads the last `seq_len` bars of past-only features plus position state and outputs one of four actions (shared mapping):

| id | action | semantics used by SP-04 (long-only by default) |
|---|---|---|
| 0 | HOLD | do nothing / stay flat / keep position |
| 1 | BUY | open a long position (ignored if already long) |
| 2 | SELL | **reduce** the long by `sell_fraction` (partial profit-taking / de-risking) |
| 3 | EXIT | close the whole position |

It also emits a **learned confidence** (probability that its own argmax is correct). Crossing a prior high is *not* enough. The model must learn to **HOLD** when evidence is weak, and **EXIT** when a breakout fails.

## Timing contract — which bars exist at decision time
A decision is made **after the close of bar `t`**. Orders fill at the **open of bar `t+1`**.

| Quantity | Bars used | Available at decision time? |
|---|---|---|
| `prior_high`, `prior_low` (range boundaries) | `t-N … t-1` (`rolling(N).max().shift(1)`) | yes. **Current bar excluded** |
| Volume average | `t-20 … t-1` (`shift(1)`) | yes |
| ATR, RSI, ADX, SMAs, returns, gap, today's volume, today's range | `≤ t` | yes (bar `t` is closed) |
| Benchmark returns | benchmark close `≤ t` | yes |
| Breakout-memory features (bars since breakout, follow-through *so far*, back-inside-range) | `≤ t` | yes. Only measures what already happened |
| Position fraction, unrealised P&L, holding bars | fills up to open of `t` | yes |
| **Anything from `t+1` onward** | | **NO. Never used in features** |

## Labels use the future, and that is explicit
* **Entry labels (BUY vs HOLD)** at a breakout-candidate bar `d` are *hindsight outcome labels*: the future bars `d+1 … d+H` decide whether the breakout achieved follow-through before failing. These bars are **only** used to build the label, never the input.
* **Management labels (HOLD / SELL / EXIT while in a position)** come from causal rules (stop, trailing stop, back-inside-range failure, profit target, time stop) evaluated on information available at that bar.
* These are **rule-derived proxy labels, not human demonstrations**. If you have genuine trader logs, replace `build_dataset()`'s label step (see the note in that cell).

## 1 · Install & environment

In [ ]:
# Colab already ships torch, numpy, pandas, scikit-learn, matplotlib. Only yfinance is needed.
!pip -q install yfinance
import sys, platform
print(sys.version)

## 2 · Configuration
Everything tunable lives in `Config`. `data_source="synthetic"` exists **only to smoke-test the pipeline offline**. Never report synthetic results.

In [ ]:
import os, sys, json, math, pickle, random, warnings, hashlib, copy, platform, zipfile, shutil
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")   # deterministic cuBLAS
from dataclasses import dataclass, field, asdict
from pathlib import Path
from types import SimpleNamespace
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from numpy.lib.stride_tricks import sliding_window_view
from sklearn.metrics import confusion_matrix, classification_report, f1_score
warnings.filterwarnings("ignore")

# ---- shared action mapping (do NOT change; must match the other SP policies) ----
HOLD, BUY, SELL, EXIT = 0, 1, 2, 3
ACTIONS = {0: "HOLD", 1: "BUY", 2: "SELL", 3: "EXIT"}
ACTION_NAMES = [ACTIONS[i] for i in range(4)]

@dataclass
class Config:
    # data
    data_source: str = "yfinance"            # "yfinance" | "synthetic" (pipeline test only)
    tickers: list = field(default_factory=lambda: ["AAPL","MSFT","AMZN","GOOGL","JPM","XOM","JNJ","PG","HD","UNH","CAT","NVDA"])
    benchmark: str = "SPY"
    start: str = "2010-01-01"
    end: str = "2025-12-31"
    train_end: str = "2019-12-31"            # train: <= train_end
    val_end: str = "2022-12-31"              # val: (train_end, val_end]; test: > val_end
    cache_dir: str = "/content/sp04_cache"
    artifact_dir: str = "/content/sp04_breakout_artifacts"
    seed: int = 42
    # sequence / features
    seq_len: int = 32
    range_lookback: int = 20                 # N: prior-range window (excludes current bar)
    breakout_memory: int = 10                # K: bars a breakout stays "active" as a feature
    # breakout candidates
    cooldown: int = 5                        # fresh breakout = first close>prior_high with none in previous `cooldown` bars
    range_max_width_atr: float = 9.0         # "established, reasonably tight range" gate: width/ATR <= this
    entry_window: int = 3                    # decision bars (fresh bar + next 2) on which entry is evaluated
    # labels (forward evaluation horizon)
    horizon: int = 10                        # H: forward bars for outcome label
    up_atr: float = 2.0                      # follow-through target (x ATR above entry)
    stop_atr: float = 1.5                    # adverse excursion that marks failure
    fail_buffer_atr: float = 0.25            # close below level - buffer*ATR => failed (back inside range)
    tp_atr: float = 3.0                      # partial profit-taking trigger (management rule -> SELL)
    trail_atr: float = 2.5                   # trailing stop from highest close (management rule -> EXIT)
    max_hold: int = 20                       # time stop (management rule -> EXIT)
    # optional trader-profile / risk features (constants unless you wire them in)
    profile_risk_tolerance: float = 0.5
    profile_horizon_pref: float = 0.5
    # model
    cell: str = "gru"                        # "gru" | "lstm"
    hidden: int = 64
    layers: int = 2
    dropout: float = 0.2
    # training
    epochs: int = 40
    batch_size: int = 256
    lr: float = 1e-3
    weight_decay: float = 1e-4
    patience: int = 8
    class_weight_power: float = 0.5          # w_c = (N/(K*n_c))**power, capped
    class_weight_cap: float = 10.0
    conf_loss_weight: float = 0.2
    feature_clip: float = 5.0
    # backtest
    initial_capital: float = 100_000.0
    fee_bps: float = 5.0                     # per side, on notional
    slippage_bps: float = 5.0                # per side, adverse
    max_positions: int = 5
    max_position_pct: float = 0.20           # of equity per name
    risk_per_trade: float = 0.01             # of equity at stop_atr distance
    sell_fraction: float = 0.5
    buy_thresholds: list = field(default_factory=lambda: [0.0, 0.35, 0.45, 0.55, 0.65])  # chosen on VALIDATION only
    min_val_trades: int = 10

CFG = Config()
Path(CFG.cache_dir).mkdir(parents=True, exist_ok=True)
Path(CFG.artifact_dir).mkdir(parents=True, exist_ok=True)
print(json.dumps(asdict(CFG), indent=1)[:600], "...")

## 3 · Reproducibility & device

In [ ]:
import torch, torch.nn as nn

def seed_everything(seed: int):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

seed_everything(CFG.seed)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", DEVICE, "| pandas", pd.__version__, "| numpy", np.__version__)

## 4 · Data download
Adjusted OHLCV from yfinance (cached to disk). Tickers are aligned to their **common trading calendar**. Note that yfinance data is a convenience source with possible survivorship bias (see limitations).

In [ ]:
def make_synthetic_ohlcv(seed, n=3900, start="2010-01-04"):
    """Regime-switching toy prices (ranges -> genuine / false / no breakout). PIPELINE TEST ONLY."""
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range(start, periods=n)
    close = np.zeros(n); vol = np.zeros(n)
    base = rng.uniform(5e5, 5e6); p = 100.0; center = p; i = 0
    def emit(ret, vm):
        nonlocal p, i
        if i >= n: return
        p *= np.exp(ret); close[i] = p; vol[i] = base * vm * rng.lognormal(0, 0.25); i += 1
    while i < n:
        for _ in range(int(rng.integers(30, 90))):
            emit(rng.normal(0, 0.008) - 0.08 * np.log(p / center), 1.0)
        u = rng.random()
        if u < 0.40:                                   # genuine breakout
            for j in range(int(rng.integers(8, 25))):
                emit(rng.normal(0.006, 0.01), 1.9 if j < 3 else 1.2)
        elif u < 0.80:                                 # false breakout
            for j in range(2): emit(rng.normal(0.015, 0.006), 1.8)
            for j in range(8): emit(rng.normal(-0.008, 0.008), 1.0)
        center = p
    prev = np.r_[close[0], close[:-1]]
    op = prev * np.exp(rng.normal(0, 0.003, n))
    hi = np.maximum(op, close) * (1 + np.abs(rng.normal(0, 0.004, n)))
    lo = np.minimum(op, close) * (1 - np.abs(rng.normal(0, 0.004, n)))
    return pd.DataFrame({"Open": op, "High": hi, "Low": lo, "Close": close, "Volume": vol.astype(np.int64)}, index=dates)

def load_ohlcv(ticker, cfg=CFG):
    if cfg.data_source == "synthetic":
        seed = int(hashlib.md5(ticker.encode()).hexdigest()[:6], 16)
        return make_synthetic_ohlcv(seed)
    path = Path(cfg.cache_dir) / f"{ticker}_{cfg.start}_{cfg.end}.csv"
    if path.exists():
        return pd.read_csv(path, index_col=0, parse_dates=True)
    import yfinance as yf
    df = yf.download(ticker, start=cfg.start, end=cfg.end, auto_adjust=True, progress=False)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df = df[["Open", "High", "Low", "Close", "Volume"]].dropna()
    df.index = pd.to_datetime(df.index).tz_localize(None)
    if len(df) < 800:
        raise ValueError(f"{ticker}: only {len(df)} rows downloaded")
    df.to_csv(path)
    return df

RAW = {}
for t in CFG.tickers:
    try:
        RAW[t] = load_ohlcv(t)
    except Exception as e:
        print(f"[skip] {t}: {e}")
BENCH = load_ohlcv(CFG.benchmark)
assert len(RAW) >= 3, "Need at least 3 tickers"
print(f"data_source={CFG.data_source} | tickers={len(RAW)} | bench rows={len(BENCH)}")
if CFG.data_source == "synthetic":
    print("!!! SYNTHETIC DATA: pipeline test only. Results are meaningless for trading. !!!")

## 5 · Feature engineering (past-only)

**Feature layout = shared base block + breakout extension block**

* `BASE` (17 cols, the shared inputs contract): 12 market features, 3 position features, 2 optional trader-profile/risk features.
* `EXT` (15 cols, breakout-specific): range geometry, compression, volatility/volume expansion, breakout-memory features.

> **Assumption to reconcile:** I do not have the other SP policies' exact input spec. `BASE` is my best-guess shared block. If your other policies use a different base vector, change `BASE_*` lists below. Nothing else needs to change, because the extension block is appended after it and the adapter (`BreakoutPolicy.build_state`) is the documented bridge.

In [ ]:
BASE_MARKET   = ["ret_1","ret_5","ret_10","ret_20","rsi_14","atr_pct","vol_rel_20","gap",
                 "sma20_50","close_vs_sma50","adx_14","rel_strength_20"]
BASE_POSITION = ["pos_frac","unrealized_pnl","holding_bars_norm"]
BASE_PROFILE  = ["profile_risk_tolerance","profile_horizon_pref"]
EXT = ["dist_prior_high_atr","dist_prior_low_atr","range_width_pct","range_width_atr","compression_ratio",
       "atr_expansion","true_range_atr","vol_rel_3","close_location","breakout_up","breakdown",
       "breakout_active","bars_since_breakout_norm","followthrough_atr","back_inside_range"]
BASE = BASE_MARKET + BASE_POSITION + BASE_PROFILE
FEATURE_COLUMNS = BASE + EXT
F_DIM, D_BASE, D_EXT = len(FEATURE_COLUMNS), len(BASE), len(EXT)
POS_SLICE = slice(len(BASE_MARKET), len(BASE_MARKET) + len(BASE_POSITION))     # contiguous by construction
PROF_SLICE = slice(POS_SLICE.stop, POS_SLICE.stop + len(BASE_PROFILE))
COL = {c: i for i, c in enumerate(FEATURE_COLUMNS)}
MKT_IDX = [i for i, c in enumerate(FEATURE_COLUMNS) if c in BASE_MARKET or c in EXT]   # scaled with train stats
print(f"F={F_DIM} (base={D_BASE}, ext={D_EXT}) | position cols {POS_SLICE} | profile cols {PROF_SLICE}")

def wilder(x, n):
    return x.ewm(alpha=1.0 / n, adjust=False, min_periods=n).mean()

def compute_feature_frame(df, bench_close, cfg):
    """
    OHLCV DataFrame -> (features[BASE_MARKET+EXT], aux). Every value at row t uses bars <= t only.
    Range boundaries use bars t-N..t-1 (shift(1)) so the current bar never defines its own breakout level.
    This function is also the documented adapter for live use: feed it recent OHLCV, take the last rows.
    """
    o, h, l, c, v = (df[x].astype(float) for x in ["Open", "High", "Low", "Close", "Volume"])
    v = v.where(v > 0)
    pc = c.shift(1)
    tr = pd.concat([h - l, (h - pc).abs(), (l - pc).abs()], axis=1).max(axis=1)
    atr = wilder(tr, 14)
    N, K = cfg.range_lookback, cfg.breakout_memory
    ph = h.rolling(N).max().shift(1)                 # prior range high  (bars t-N..t-1)
    pl = l.rolling(N).min().shift(1)                 # prior range low
    width = ph - pl
    vol_avg = v.rolling(20).mean().shift(1)          # average volume over t-20..t-1
    f = pd.DataFrame(index=df.index)
    lc = np.log(c)
    for n in (1, 5, 10, 20):
        f[f"ret_{n}"] = lc.diff(n)
    d = c.diff()
    rsi = 100 - 100 / (1 + wilder(d.clip(lower=0), 14) / wilder((-d).clip(lower=0), 14))
    f["rsi_14"] = (rsi - 50) / 50
    f["atr_pct"] = atr / c
    f["vol_rel_20"] = np.log(v / vol_avg).clip(-3, 3)
    f["gap"] = o / pc - 1                            # today's open is known at today's close
    sma20, sma50 = c.rolling(20).mean(), c.rolling(50).mean()
    f["sma20_50"] = sma20 / sma50 - 1
    f["close_vs_sma50"] = c / sma50 - 1
    up_m, dn_m = h.diff(), -l.diff()
    pdm = pd.Series(np.where((up_m > dn_m) & (up_m > 0), up_m, 0.0), index=df.index)
    mdm = pd.Series(np.where((dn_m > up_m) & (dn_m > 0), dn_m, 0.0), index=df.index)
    pdi, mdi = 100 * wilder(pdm, 14) / atr, 100 * wilder(mdm, 14) / atr
    dx = 100 * (pdi - mdi).abs() / (pdi + mdi).replace(0, np.nan)
    f["adx_14"] = wilder(dx, 14) / 100
    bret = np.log(bench_close.astype(float)).diff(20).reindex(df.index).ffill()
    f["rel_strength_20"] = f["ret_20"] - bret
    # ---- breakout extension block ----
    f["dist_prior_high_atr"] = (c - ph) / atr
    f["dist_prior_low_atr"] = (c - pl) / atr
    f["range_width_pct"] = width / c
    f["range_width_atr"] = width / atr
    f["compression_ratio"] = (width / c) / (width / c).rolling(120).mean()
    f["atr_expansion"] = np.log(tr.rolling(5).mean() / tr.rolling(30).mean())
    f["true_range_atr"] = tr / atr
    f["vol_rel_3"] = np.log(v.rolling(3).mean() / vol_avg).clip(-3, 3)
    f["close_location"] = (((c - l) / (h - l).replace(0, np.nan)).fillna(0.5)) * 2 - 1
    bu = ((c > ph) & ph.notna()).astype(float)
    f["breakout_up"] = bu
    f["breakdown"] = ((c < pl) & pl.notna()).astype(float)
    recent = bu.shift(1).rolling(cfg.cooldown, min_periods=1).max().fillna(0) > 0
    fresh = (bu > 0) & ~recent                                   # first breakout of a cluster
    idx = pd.Series(np.arange(len(c), dtype=float), index=c.index)
    since = idx - idx.where(fresh).ffill()                       # bars since last fresh breakout (past only)
    level = ph.where(fresh).ffill()                              # breakout level frozen at the fresh bar
    rok_pt = (width / atr) <= cfg.range_max_width_atr
    rok_lvl = rok_pt.astype(float).where(fresh).ffill() > 0.5
    active = since.notna() & (since <= K)
    f["breakout_active"] = active.astype(float)
    f["bars_since_breakout_norm"] = (since / K).where(active, 1.0)
    f["followthrough_atr"] = ((c - level) / atr).where(active, 0.0)
    f["back_inside_range"] = ((c < level) & active).astype(float)
    cand_dec = active & (since < cfg.entry_window) & rok_lvl & (c > level)
    aux = pd.DataFrame({"O": o, "H": h, "L": l, "C": c, "atr": atr, "level": level,
                        "cand_dec": cand_dec, "fresh": fresh, "fresh_cand": fresh & rok_pt,
                        "exit_low10": l.rolling(10).min().shift(1)}, index=df.index)
    return f[BASE_MARKET + EXT], aux

### 5.1 No-look-ahead unit test
For random bars `t`, features computed from data truncated at `t` must equal features at `t` computed from the full history. If any feature peeked at the future this fails.

In [ ]:
def assert_no_lookahead(df, bench_close, cfg, n_checks=8, seed=0):
    full, _ = compute_feature_frame(df, bench_close, cfg)
    rng = np.random.default_rng(seed)
    for t in rng.integers(300, len(df) - 5, n_checks):
        part, _ = compute_feature_frame(df.iloc[: t + 1], bench_close.loc[: df.index[t]], cfg)
        a, b = full.iloc[t].values, part.iloc[-1].values
        ok = np.allclose(a, b, equal_nan=True, rtol=1e-9, atol=1e-9)
        assert ok, f"LOOK-AHEAD at t={t}: {full.columns[~np.isclose(a, b, equal_nan=True)].tolist()}"
    # perturbing future bars must not change today's features
    t = len(df) // 2
    df2 = df.copy(); df2.iloc[t + 1:, :4] *= 1.37
    p2, _ = compute_feature_frame(df2, bench_close, cfg)
    assert np.allclose(full.iloc[t].values, p2.iloc[t].values, equal_nan=True), "future perturbation leaked"
    print("no-look-ahead test passed")

_t0 = next(iter(RAW))
assert_no_lookahead(RAW[_t0], BENCH["Close"], CFG)

## 6 · Aligned panel
All tickers are cut to the post-warm-up common calendar and stacked into `(D bars, K tickers, F features)`. Position columns start at zero and profile columns are constants. Position columns are filled per sample (training) or by the simulator (backtest).

In [ ]:
def build_panel(raw, bench, cfg):
    parts = {}
    for t, df in raw.items():
        feat, aux = compute_feature_frame(df, bench["Close"], cfg)
        feat = feat.replace([np.inf, -np.inf], np.nan)
        valid = feat.notna().all(axis=1) & aux["atr"].notna()
        if valid.sum() < 600:
            print(f"[skip] {t}: too little valid history"); continue
        first = valid.idxmax()
        parts[t] = (feat.loc[first:], aux.loc[first:])
    common = None
    for t, (f, a) in parts.items():
        common = f.index if common is None else common.intersection(f.index)
    common = common.sort_values()
    tickers = list(parts)
    D, K = len(common), len(tickers)
    P = SimpleNamespace(dates=common, tickers=tickers, D=D, K=K)
    P.X = np.zeros((D, K, F_DIM), dtype=np.float32)
    for name in ["O","H","L","C","atr","level","exit_low10"]:
        setattr(P, name, np.zeros((D, K)))
    for name in ["cand_dec","fresh","fresh_cand"]:
        setattr(P, name, np.zeros((D, K), dtype=bool))
    for k, t in enumerate(tickers):
        f, a = parts[t]
        f, a = f.reindex(common).fillna(0.0), a.reindex(common)
        P.X[:, k, [COL[c] for c in BASE_MARKET + EXT]] = f[BASE_MARKET + EXT].values
        for name in ["O","H","L","C","atr","level","exit_low10"]:
            getattr(P, name)[:, k] = a[name].values
        for name in ["cand_dec","fresh","fresh_cand"]:
            getattr(P, name)[:, k] = a[name].fillna(False).values.astype(bool)
    P.X[:, :, PROF_SLICE] = np.array([cfg.profile_risk_tolerance, cfg.profile_horizon_pref], dtype=np.float32)
    b = bench.reindex(common).ffill()
    P.bench_O, P.bench_C = b["Open"].values, b["Close"].values
    return P

PANEL = build_panel(RAW, BENCH, CFG)
print(f"panel: {PANEL.D} bars x {PANEL.K} tickers | {PANEL.dates[0].date()} -> {PANEL.dates[-1].date()}")
d_ = PANEL.dates
split_of_date = lambda dt: 0 if dt <= pd.Timestamp(CFG.train_end) else (1 if dt <= pd.Timestamp(CFG.val_end) else 2)
SPLIT_NAMES = ["train", "val", "test"]
SPLIT_IDX = np.array([split_of_date(x) for x in d_])           # split of each bar index
assert (SPLIT_IDX == 0).sum() > CFG.seq_len * 5 and (SPLIT_IDX == 1).sum() > 100 and (SPLIT_IDX == 2).sum() > 100, "check split dates"
print({n: int((SPLIT_IDX == i).sum()) for i, n in enumerate(SPLIT_NAMES)}, "bars per split")

## 7 · Label construction

### 7.1 Breakout candidates (past-only definition)
A **fresh breakout** at bar `t0`: `close[t0] > prior_high[t0]` (range of `t0-N … t0-1`) with no raw breakout in the previous `cooldown` bars. It is a **candidate** only if the prior range was *established and reasonably tight*: `(prior_high-prior_low)/ATR <= range_max_width_atr`. The *entry decision bars* are `t0 … t0+entry_window-1`, as long as the close is still above the frozen breakout level, so the model can use early follow-through evidence. This definition uses only information available at the decision bar.

### 7.2 Outcome label (**uses future bars. Label only, never an input**)
For decision bar `d`, entry = open of `d+1`, `ATR_d`, frozen `level`. Scan bars `d+1 … d+H` in order:

* **false breakout (0)**: first the low touches `entry − stop_atr·ATR` *or* a close falls below `level − fail_buffer·ATR` (back inside the range). Failure is checked **before** the target in the same bar (conservative).
* **true breakout (1)**: the high reaches `entry + up_atr·ATR` first.
* **unconfirmed (2)**: neither within `H` bars. Treated as *not a valid opportunity* → HOLD.

Flat-state label: **BUY only if true**; everything else, including all non-candidates, false breakouts and unconfirmed ones, is **HOLD**. The label builder never forces a trade where no valid opportunity exists. Candidates whose forward window is cut off by the end of data are dropped.

### 7.3 Management labels (causal rules, in-position rows)
For every candidate bar `t0` we simulate a **shadow position** entered naively at the open of `t0+1`, *regardless of the later outcome*. That is what gives the model real examples of a failing breakout while holding it. On each bar `s` the proxy "trader" decides using only information up to `s`:
`EXIT` if close < level − buffer·ATR (failed / back in range), or close ≤ entry − stop_atr·ATR, or close ≤ highest_close − trail_atr·ATR, or held ≥ `max_hold`; else `SELL` (once, reduce half) if close ≥ entry + tp_atr·ATR; else `HOLD`.

> **These are rule-derived proxy labels, not human demonstrations.** To use genuine trader labels, replace the `y` assigned in `build_dataset()` (flat rows) and `simulate_trajectory()` (in-position rows) with your logged actions, keeping the same `X` construction.

In [ ]:
# candidate outcome label panel: -1 none, 0 false, 1 true, 2 unconfirmed, -2 undetermined (future cut off)
def eval_candidate(d, k, P, cfg):
    e, end = d + 1, d + cfg.horizon
    if end >= P.D:
        return -2
    entry, a, lvl = P.O[e, k], P.atr[d, k], P.level[d, k]
    target, stop, fail_px = entry + cfg.up_atr * a, entry - cfg.stop_atr * a, lvl - cfg.fail_buffer_atr * a
    for s in range(e, end + 1):                       # FUTURE bars: used for the LABEL only
        if P.L[s, k] <= stop or P.C[s, k] < fail_px:
            return 0
        if P.H[s, k] >= target:
            return 1
    return 2

def build_cand_labels(P, cfg):
    lab = np.full((P.D, P.K), -1, dtype=np.int8)
    for k in range(P.K):
        for d in np.where(P.cand_dec[:, k])[0]:
            lab[d, k] = eval_candidate(d, k, P, cfg)
    return lab

PANEL.cand_label = build_cand_labels(PANEL, CFG)
CAT_NAMES = {-2: "undetermined", -1: "non_candidate", 0: "false_breakout", 1: "true_breakout", 2: "unconfirmed"}
for si, sn in enumerate(SPLIT_NAMES):
    m = SPLIT_IDX == si
    cl = PANEL.cand_label[m]
    print(sn, {CAT_NAMES[v]: int((cl == v).sum()) for v in (0, 1, 2)}, "| fresh candidates:", int(PANEL.fresh_cand[m].sum()))
if PANEL.fresh_cand.sum() < 100:
    print("WARNING: few candidates. Consider more tickers, a longer history or looser range_max_width_atr.")

In [ ]:
def simulate_trajectory(t0, k, P, cfg):
    """Shadow position entered at open[t0+1]; management labels from CAUSAL rules (info <= s). Returns rows or None."""
    e = t0 + 1
    if e >= P.D:
        return None
    entry, a, lvl = P.O[e, k], P.atr[t0, k], P.level[t0, k]
    frac, max_c, rows = 1.0, entry, []
    for s in range(e, min(P.D, e + cfg.max_hold)):
        held, c = s - e + 1, P.C[s, k]
        max_c = max(max_c, c)
        pnl = c / entry - 1.0
        fail = c < lvl - cfg.fail_buffer_atr * a
        stop = c <= entry - cfg.stop_atr * a
        trail = c <= max_c - cfg.trail_atr * a
        if fail or stop or trail or held >= cfg.max_hold:
            rows.append((s, frac, pnl, held / cfg.max_hold, EXIT)); break
        if frac == 1.0 and c >= entry + cfg.tp_atr * a:
            rows.append((s, frac, pnl, held / cfg.max_hold, SELL)); frac = 0.5
        else:
            rows.append((s, frac, pnl, held / cfg.max_hold, HOLD))
    return SimpleNamespace(e=e, rows=rows)

def build_dataset(P, cfg):
    """Returns X_raw (N,L,F) (unscaled, with position columns), y, and metadata. Chronological purge applied."""
    L = cfg.seq_len
    rk, rt, ry, rseg, rend, rtraj = [], [], [], [], [], []
    seg_of = {1: 1, 0: 2, 2: 3}                              # cand label -> segment id
    for k in range(P.K):                                      # flat-state rows (every bar is a decision point)
        for t in range(L - 1, P.D):
            lab = int(P.cand_label[t, k])
            if lab == -2:
                continue
            if lab == -1:
                rk.append(k); rt.append(t); ry.append(HOLD); rseg.append(0); rend.append(t); rtraj.append(-1)
            else:
                rk.append(k); rt.append(t); ry.append(BUY if lab == 1 else HOLD); rseg.append(seg_of[lab])
                rend.append(t + cfg.horizon); rtraj.append(-1)
    trajs = []
    for k in range(P.K):                                      # in-position rows from shadow positions
        for t0 in np.where(P.fresh_cand[:, k])[0]:
            tr = simulate_trajectory(t0, k, P, cfg)
            if tr is None or not tr.rows:
                continue
            tr.pos = np.array([[r[1], r[2], r[3]] for r in tr.rows], dtype=np.float32)
            trajs.append(tr)
            for j, (s, fr, pnl, hn, lab) in enumerate(tr.rows):
                if s < L - 1:
                    continue
                rk.append(k); rt.append(s); ry.append(lab); rseg.append(4); rend.append(s); rtraj.append(len(trajs) - 1)
            tr.k = k
    k_a, t_a, y_a, seg_a, end_a, tj_a = map(np.array, (rk, rt, ry, rseg, rend, rtraj))
    # chronological purge: label horizon must end inside the same split as the decision bar
    keep = SPLIT_IDX[t_a] == SPLIT_IDX[np.minimum(end_a, P.D - 1)]
    k_a, t_a, y_a, seg_a, end_a, tj_a = (z[keep] for z in (k_a, t_a, y_a, seg_a, end_a, tj_a))
    sw = sliding_window_view(P.X, L, axis=0)                  # (D-L+1, K, F, L)
    X = np.ascontiguousarray(sw[t_a - L + 1, k_a].transpose(0, 2, 1)).astype(np.float32)
    for r in np.where(tj_a >= 0)[0]:                          # overwrite position columns with the shadow position's state
        tr, s = trajs[tj_a[r]], t_a[r]
        lo = max(tr.e, s - L + 1)
        X[r, lo - (s - L + 1):, POS_SLICE] = tr.pos[lo - tr.e: s - tr.e + 1]
    meta = SimpleNamespace(k=k_a, t=t_a, seg=seg_a, split=SPLIT_IDX[t_a], dates=P.dates[t_a])
    return X, y_a.astype(np.int64), meta

X_RAW, Y, META = build_dataset(PANEL, CFG)
SEG_NAMES = {0: "flat_non_candidate", 1: "candidate_true", 2: "candidate_false", 3: "candidate_unconfirmed", 4: "in_position"}
print("samples:", X_RAW.shape, f"~{X_RAW.nbytes/1e6:.0f} MB")
for si, sn in enumerate(SPLIT_NAMES):
    m = META.split == si
    print(f"{sn:5s} n={m.sum():6d}", {ACTIONS[a]: int((Y[m] == a).sum()) for a in range(4)})
# split hygiene
for si in range(2):
    assert PANEL.dates[META.t[META.split == si]].max() < PANEL.dates[META.t[META.split == si + 1]].min()
print("chronological split check passed")

## 8 · Preprocessing (fit on training data only)
Per-feature z-score statistics are computed from **training-period panel rows only** for market/extension columns, then clipped to `±feature_clip`. Position and profile columns use *fixed* scaling (no fitting). Everything is stored as plain lists/arrays so loading never needs the data or sklearn.

In [ ]:
def fit_preprocessing(P, cfg):
    train_rows = P.X[SPLIT_IDX == 0]                                   # (n_train_bars, K, F) -> training period only
    flat = train_rows.reshape(-1, F_DIM)
    mean, std = np.zeros(F_DIM), np.ones(F_DIM)
    mean[MKT_IDX] = flat[:, MKT_IDX].mean(0)
    std[MKT_IDX] = flat[:, MKT_IDX].std(0)
    std[std < 1e-8] = 1.0
    mean[POS_SLICE], std[POS_SLICE] = [0.0, 0.0, 0.0], [1.0, 0.1, 1.0]  # fixed: frac, pnl (10% = 1 unit), hold/max_hold
    mean[PROF_SLICE], std[PROF_SLICE] = 0.5, 0.5                        # fixed
    ext_neutral = mean[[COL[c] for c in EXT]].copy()                    # raw-space "neutral" values for the adapter
    return dict(mean=mean.tolist(), std=std.tolist(), clip=cfg.feature_clip, columns=FEATURE_COLUMNS,
                fitted_on=f"{P.dates[SPLIT_IDX == 0][0].date()}..{P.dates[SPLIT_IDX == 0][-1].date()}",
                ext_neutral_raw=ext_neutral.tolist(), mkt_idx=MKT_IDX)

PREPROC = fit_preprocessing(PANEL, CFG)

def transform(x, pre):
    m, s = np.asarray(pre["mean"], np.float32), np.asarray(pre["std"], np.float32)
    return np.clip((x - m) / s, -pre["clip"], pre["clip"]).astype(np.float32)

X = transform(X_RAW, PREPROC)
tr_m, va_m, te_m = (META.split == 0), (META.split == 1), (META.split == 2)
Xtr, ytr, Xva, yva, Xte, yte = X[tr_m], Y[tr_m], X[va_m], Y[va_m], X[te_m], Y[te_m]
print("scaled shapes:", Xtr.shape, Xva.shape, Xte.shape, "| train mean|z| =", float(np.abs(Xtr).mean()))

## 9 · Model: GRU/LSTM sequence classifier with action head + confidence head
* Input: `(batch, seq_len, F)` scaled features. Output: 4 action logits + 1 confidence logit.
* **Confidence** = a small head (fed a *detached* hidden state) trained with BCE to predict "is my argmax correct?". It does not influence the action head.
* **Class imbalance**: class-weighted cross-entropy (`(N/(K·n_c))^power`, capped). BUY / SELL / EXIT are rare relative to HOLD.

In [ ]:
class BreakoutNet(nn.Module):
    def __init__(self, n_features, hidden=64, layers=2, dropout=0.2, cell="gru", n_actions=4):
        super().__init__()
        self.cfg = dict(n_features=n_features, hidden=hidden, layers=layers, dropout=dropout, cell=cell, n_actions=n_actions)
        self.inp = nn.Sequential(nn.Linear(n_features, hidden), nn.GELU())
        rnn = nn.GRU if cell == "gru" else nn.LSTM
        self.rnn = rnn(hidden, hidden, num_layers=layers, batch_first=True, dropout=dropout if layers > 1 else 0.0)
        self.norm = nn.LayerNorm(hidden)
        self.head = nn.Sequential(nn.Linear(hidden, hidden), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden, n_actions))
        self.conf = nn.Sequential(nn.Linear(hidden, hidden // 2), nn.GELU(), nn.Linear(hidden // 2, 1))
    def forward(self, x):
        out, _ = self.rnn(self.inp(x))
        h = self.norm(out[:, -1])
        return self.head(h), self.conf(h.detach()).squeeze(-1)

def class_weights(y, cfg, n_cls=4):
    cnt = np.bincount(y, minlength=n_cls).astype(float)
    w = np.ones(n_cls)
    for c in range(n_cls):
        if cnt[c] > 0:
            w[c] = min((len(y) / (n_cls * cnt[c])) ** cfg.class_weight_power, cfg.class_weight_cap)
    return torch.tensor(w, dtype=torch.float32), cnt

@torch.no_grad()
def predict_proba(model, X, device, bs=4096):
    model.eval(); P_, C_ = [], []
    for i in range(0, len(X), bs):
        lg, cf = model(torch.from_numpy(X[i:i + bs]).to(device))
        P_.append(torch.softmax(lg, -1).cpu().numpy()); C_.append(torch.sigmoid(cf).cpu().numpy())
    return np.concatenate(P_), np.concatenate(C_)

def macro_f1_present(y, pred):
    labs = sorted(set(y.tolist()))
    return f1_score(y, pred, labels=labs, average="macro", zero_division=0)

def train_model(Xtr, ytr, Xva, yva, cfg, device):
    seed_everything(cfg.seed)
    model = BreakoutNet(Xtr.shape[-1], cfg.hidden, cfg.layers, cfg.dropout, cfg.cell).to(device)
    w, cnt = class_weights(ytr, cfg)
    print("train class counts:", dict(zip(ACTION_NAMES, cnt.astype(int))), "| weights:", np.round(w.numpy(), 2))
    ce = nn.CrossEntropyLoss(weight=w.to(device)); bce = nn.BCEWithLogitsLoss()
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=3)
    Xt, yt = torch.from_numpy(Xtr).to(device), torch.from_numpy(ytr).to(device)
    Xv, yv = torch.from_numpy(Xva).to(device), torch.from_numpy(yva).to(device)
    gen = torch.Generator().manual_seed(cfg.seed)
    hist, best, best_state, bad = [], -1.0, None, 0
    for ep in range(1, cfg.epochs + 1):
        model.train(); perm = torch.randperm(len(Xt), generator=gen).to(device); tot = 0.0
        for i in range(0, len(perm), cfg.batch_size):
            idx = perm[i:i + cfg.batch_size]; xb, yb = Xt[idx], yt[idx]
            lg, cf = model(xb)
            loss = ce(lg, yb) + cfg.conf_loss_weight * bce(cf, (lg.argmax(-1) == yb).float().detach())
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            tot += loss.item() * len(idx)
        model.eval()
        with torch.no_grad():
            lgv, _ = model(Xv); vloss = ce(lgv, yv).item(); vpred = lgv.argmax(-1).cpu().numpy()
        vf1 = macro_f1_present(yva, vpred)
        sched.step(vf1)
        hist.append(dict(epoch=ep, train_loss=tot / len(Xt), val_loss=vloss, val_macro_f1=vf1))
        print(f"ep {ep:02d} train {tot/len(Xt):.4f} | val loss {vloss:.4f} | val macroF1(present) {vf1:.4f}")
        if vf1 > best + 1e-4:
            best, bad = vf1, 0; best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= cfg.patience:
                print(f"early stopping at epoch {ep} (best val macroF1 {best:.4f})"); break
    model.load_state_dict(best_state)
    return model, pd.DataFrame(hist), w.numpy().tolist()

MODEL, HIST, CLASS_W = train_model(Xtr, ytr, Xva, yva, CFG, DEVICE)

## 10 · `BreakoutPolicy`: inference interface
Contract (identical across SP policies, as far as I can infer):

```python
policy = BreakoutPolicy.load("breakout_policy.pt", device="cpu")
out = policy.predict(state)   # state: raw (unscaled) ndarray (T, F) in schema order, or {"base":(T,17), "extra":(T,15)}
out -> {"action": int, "action_name": str, "probabilities": [p_HOLD,p_BUY,p_SELL,p_EXIT],
        "probabilities_by_action": {...}, "confidence": float, "max_prob": float, "raw_argmax": int, "warnings": [...], ...}
```
* The `.pt` embeds weights, scaler, schema and decision rule, so **loading needs no downloads, sklearn or pickle** (`weights_only=True`).
* **Adapter for breakout-specific features:** `policy.build_state(base, extra=None)` concatenates the shared base block with the 15 extension columns. If the caller has no extension features, they are filled with training means (scaled value ≈ 0, i.e. "no breakout information") and a warning is returned. Expect weaker, *neutral* behaviour in that case. For raw OHLCV, use `compute_feature_frame()` from §5.
* `buy_threshold`: a BUY whose probability is below this value is downgraded to HOLD (avoids forcing trades). It is chosen on validation in §13.

In [ ]:
class BreakoutPolicy:
    NAME, VERSION = "SP-04 Breakout Policy", "1.0"
    ACTIONS = ACTIONS

    def __init__(self, net, payload, device="cpu"):
        self.device = torch.device(device)
        self.net = net.to(self.device).eval()
        self.payload = payload
        self.columns = list(payload["feature_columns"])
        self.seq_len = int(payload["seq_len"])
        self.mean = torch.tensor(payload["preproc"]["mean"], dtype=torch.float32)
        self.std = torch.tensor(payload["preproc"]["std"], dtype=torch.float32)
        self.clip = float(payload["preproc"]["clip"])
        self.buy_threshold = float(payload["decision"]["buy_threshold"])
        self.d_base, self.d_ext = int(payload["groups"]["d_base"]), int(payload["groups"]["d_ext"])
        self.ext_neutral = np.asarray(payload["preproc"]["ext_neutral_raw"], dtype=np.float32)

    # ---- adapter -------------------------------------------------------------
    def build_state(self, base, extra=None):
        base = np.asarray(base, dtype=np.float32)
        warn = []
        if base.ndim != 2 or base.shape[1] != self.d_base:
            raise ValueError(f"base must be (T,{self.d_base}) in order {self.columns[:self.d_base]}")
        if extra is None:
            extra = np.tile(self.ext_neutral, (base.shape[0], 1))
            warn.append("extension features missing: filled with training means (neutral)")
        extra = np.asarray(extra, dtype=np.float32)
        if extra.shape != (base.shape[0], self.d_ext):
            raise ValueError(f"extra must be ({base.shape[0]},{self.d_ext})")
        return np.concatenate([base, extra], axis=1), warn

    def _coerce(self, state):
        warn = []
        if isinstance(state, dict):
            state, warn = self.build_state(state["base"], state.get("extra"))
        x = np.asarray(state, dtype=np.float32)
        if x.ndim == 1:
            x = x[None, :]
        if x.ndim != 2 or x.shape[1] != len(self.columns):
            raise ValueError(f"state must be (T,{len(self.columns)}); got {x.shape}")
        if not np.isfinite(x).all():
            raise ValueError("state contains NaN/inf")
        if x.shape[0] < self.seq_len:
            warn.append(f"history shorter than seq_len={self.seq_len}: left-padded with first row")
            x = np.concatenate([np.repeat(x[:1], self.seq_len - x.shape[0], axis=0), x], axis=0)
        return x[-self.seq_len:], warn

    # ---- inference -----------------------------------------------------------
    @torch.no_grad()
    def predict_batch(self, windows):
        """windows: raw (B,T=seq_len,F). Returns dict of arrays: probs (B,4), action, raw_argmax, confidence, max_prob."""
        w = torch.as_tensor(np.asarray(windows, dtype=np.float32))
        if w.ndim != 3 or w.shape[1] != self.seq_len or w.shape[2] != len(self.columns):
            raise ValueError(f"windows must be (B,{self.seq_len},{len(self.columns)}); got {tuple(w.shape)}")
        if not torch.isfinite(w).all():
            raise ValueError("windows contain NaN/inf")
        z = torch.clamp((w - self.mean) / self.std, -self.clip, self.clip).to(self.device)
        self.net.eval()
        logits, cf = self.net(z)
        probs = torch.softmax(logits, -1).cpu().numpy()
        raw = probs.argmax(1)
        action = raw.copy()
        action[(raw == BUY) & (probs[:, BUY] < self.buy_threshold)] = HOLD     # do not force low-conviction trades
        return dict(probs=probs, raw_argmax=raw, action=action,
                    confidence=torch.sigmoid(cf).cpu().numpy(), max_prob=probs.max(1))

    def predict(self, state):
        x, warn = self._coerce(state)
        o = self.predict_batch(x[None])
        a = int(o["action"][0]); p = [float(v) for v in o["probs"][0]]
        return {"policy": self.NAME, "version": self.VERSION, "action": a, "action_name": ACTIONS[a],
                "probabilities": p, "probabilities_by_action": {ACTIONS[i]: p[i] for i in range(4)},
                "confidence": float(o["confidence"][0]), "max_prob": float(o["max_prob"][0]),
                "raw_argmax": int(o["raw_argmax"][0]), "action_probability": p[a], "warnings": warn}

    # ---- persistence ---------------------------------------------------------
    def save(self, path):
        payload = dict(self.payload)
        payload["decision"] = {"buy_threshold": float(self.buy_threshold)}
        payload["state_dict"] = {k: v.detach().cpu() for k, v in self.net.state_dict().items()}
        Path(path).parent.mkdir(parents=True, exist_ok=True)
        torch.save(payload, str(path))
        return str(path)

    @classmethod
    def load(cls, path, device="cpu"):
        payload = torch.load(str(path), map_location="cpu", weights_only=True)   # tensors + primitives only
        assert payload["format"] == "sp04_breakout_policy_v1", "unexpected artifact format"
        net = BreakoutNet(**payload["model_config"])
        net.load_state_dict(payload["state_dict"])
        return cls(net, payload, device)

    @classmethod
    def from_training(cls, model, cfg, preproc, buy_threshold=0.0, device="cpu"):
        payload = dict(format="sp04_breakout_policy_v1", policy=cls.NAME, version=cls.VERSION,
                       model_config=model.cfg, feature_columns=FEATURE_COLUMNS, seq_len=cfg.seq_len,
                       groups=dict(d_base=D_BASE, d_ext=D_EXT, base=BASE, ext=EXT,
                                   position=BASE_POSITION, profile=BASE_PROFILE),
                       preproc={k: preproc[k] for k in ["mean", "std", "clip", "ext_neutral_raw", "fitted_on"]},
                       actions={str(k): v for k, v in ACTIONS.items()},
                       decision=dict(buy_threshold=float(buy_threshold)), seed=cfg.seed, torch_version=str(torch.__version__))
        payload["state_dict"] = {k: v.detach().cpu() for k, v in model.state_dict().items()}
        net = BreakoutNet(**model.cfg); net.load_state_dict(payload["state_dict"])
        return cls(net, payload, device)

POLICY = BreakoutPolicy.from_training(MODEL, CFG, PREPROC, buy_threshold=0.0, device=DEVICE)
print("BreakoutPolicy ready | action map:", ACTIONS)

## 11 · Classification evaluation (validation & held-out test)
Held-out **test** metrics are computed once the decision rule is frozen (§13). Here, validation and test classification with plain `argmax`.

In [ ]:
def cls_eval(y, probs, thr=0.0):
    pred = probs.argmax(1).copy()
    pred[(pred == BUY) & (probs[:, BUY] < thr)] = HOLD
    labs = [0, 1, 2, 3]
    cm = confusion_matrix(y, pred, labels=labs)
    rep = classification_report(y, pred, labels=labs, target_names=ACTION_NAMES, output_dict=True, zero_division=0)
    return pred, cm, rep

def segment_report(y, pred, seg):
    out = {}
    for sid, name in SEG_NAMES.items():
        m = seg == sid
        if m.sum() == 0:
            out[name] = dict(n=0); continue
        d = dict(n=int(m.sum()), accuracy=float((pred[m] == y[m]).mean()),
                 pred_dist={ACTIONS[a]: float((pred[m] == a).mean()) for a in range(4)},
                 label_dist={ACTIONS[a]: float((y[m] == a).mean()) for a in range(4)})
        if sid in (1, 2, 3):
            d["buy_rate"] = float((pred[m] == BUY).mean())
        if sid == 4:
            d["exit_recall"] = float(((pred[m] == EXIT) & (y[m] == EXIT)).sum() / max(1, (y[m] == EXIT).sum()))
            d["exit_precision"] = float(((pred[m] == EXIT) & (y[m] == EXIT)).sum() / max(1, (pred[m] == EXIT).sum()))
        out[name] = d
    return out

def plot_cm(cm, title):
    fig, ax = plt.subplots(figsize=(4.6, 4))
    cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    for i in range(4):
        for j in range(4):
            ax.text(j, i, f"{cm[i, j]}\n({cmn[i, j]:.0%})", ha="center", va="center", fontsize=8, color="black" if cmn[i, j] < .6 else "white")
    ax.set_xticks(range(4)); ax.set_xticklabels(ACTION_NAMES); ax.set_yticks(range(4)); ax.set_yticklabels(ACTION_NAMES)
    ax.set_xlabel("predicted"); ax.set_ylabel("proxy label"); ax.set_title(title); plt.tight_layout(); plt.show()

pva, cva = predict_proba(MODEL, Xva, DEVICE)
pred_va, cm_va, rep_va = cls_eval(yva, pva)
print("VALIDATION (argmax)\n", classification_report(yva, pred_va, labels=[0, 1, 2, 3], target_names=ACTION_NAMES, zero_division=0))
plot_cm(cm_va, "Validation confusion (argmax)")
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
HIST.plot(x="epoch", y=["train_loss", "val_loss"], ax=ax[0], title="loss"); HIST.plot(x="epoch", y="val_macro_f1", ax=ax[1], title="val macro-F1 (present classes)")
plt.tight_layout(); plt.show()

## 12 · Backtest engine (next-bar execution, costs, limits)
* Decision after the close of bar `d` → order fills at the **open of `d+1`** (never the same bar).
* Costs: `fee_bps` + `slippage_bps` per side, applied adversely to the fill price. Whole shares only.
* Limits: at most `max_positions` open names, each ≤ `max_position_pct` of equity; size is also capped so that a `stop_atr·ATR` move loses ≤ `risk_per_trade` of equity. Orders are limited by available cash. No leverage, no shorting.
* `SELL` sells `sell_fraction` of current shares; `EXIT` sells all; `BUY` is ignored when already long. When more BUY signals than free slots arrive on the same bar, slots are filled in ticker-list order (the policy returns actions only, not rankings).
* The model sees **its own simulated position state** (fraction, unrealised P&L, holding bars) over the window, exactly like in training.
* `turnover` = total traded notional over the window / mean equity (not annualised). `trade_count` = closed round trips (positions still open at the end are marked to market and counted separately).
* The same engine runs the baselines. Buy-and-hold uses the same cost model and the same first-fill timing (open of the first bar after the start).

In [ ]:
def make_model_policy(policy, P, cfg):
    L = cfg.seq_len
    def fn(d, pos_hist):
        win = P.X[d - L + 1: d + 1].transpose(1, 0, 2).copy()               # (K, L, F) raw
        win[:, :, POS_SLICE] = pos_hist[d - L + 1: d + 1].transpose(1, 0, 2)
        return policy.predict_batch(win)["action"]
    return fn

def make_breakout_baseline(P, cfg, vol_confirm=False):
    """Donchian-style rule: BUY the first close above the prior N-bar high (optionally needing volume >= 1.5x avg),
    EXIT on a close below the prior 10-bar low. No range filter, no learned evidence."""
    thr = np.log(1.5)
    def fn(d, pos_hist):
        a = np.zeros(P.K, dtype=int); holding = pos_hist[d, :, 0] > 0
        entry = P.fresh[d] & ~holding
        if vol_confirm:
            entry &= P.X[d, :, COL["vol_rel_20"]] >= thr
        a[entry] = BUY
        a[holding & (P.C[d] < P.exit_low10[d])] = EXIT
        return a
    return fn

def backtest(P, policy_fn, i0, i1, cfg):
    K = P.K; fee, slip = cfg.fee_bps / 1e4, cfg.slippage_bps / 1e4
    cash = cfg.initial_capital
    shares, avg, full = np.zeros(K), np.zeros(K), np.zeros(K)
    ebar, sig = np.full(K, -1), np.full(K, -1)
    cost, proceeds = np.zeros(K), np.zeros(K)
    pos_hist = np.zeros((P.D, K, 3), dtype=np.float32)
    pending, pstop = np.zeros(K, dtype=int), np.zeros(K)
    eq, expo, trades, traded, n_orders = [], [], [], 0.0, 0

    def record(k, d, exit_idx, proceeds_total, open_end=False):
        pnl = proceeds_total - cost[k]
        trades.append(dict(ticker=P.tickers[k], entry_date=P.dates[ebar[k]], exit_date=P.dates[exit_idx],
                           hold_bars=int(exit_idx - ebar[k]), cost=cost[k], proceeds=proceeds_total, pnl=pnl,
                           ret=proceeds_total / cost[k] - 1.0, signal_idx=int(sig[k]),
                           category=CAT_NAMES[int(P.cand_label[sig[k], k])] if sig[k] >= 0 else "n/a",
                           open_at_end=open_end))

    for d in range(i0, i1 + 1):
        if d > i0 and pending.any():
            eq_open = cash + (shares * P.O[d]).sum()
            for k in np.where((pending == SELL) | (pending == EXIT))[0]:      # free cash first
                if shares[k] <= 0:
                    continue
                qty = shares[k] if pending[k] == EXIT else min(shares[k], max(1.0, math.floor(shares[k] * cfg.sell_fraction)))
                px = P.O[d, k] * (1 - slip); notional = qty * px; f = notional * fee
                cash += notional - f; shares[k] -= qty; proceeds[k] += notional - f; traded += notional; n_orders += 1
                if shares[k] <= 1e-9:
                    shares[k] = 0.0; record(k, d, d, proceeds[k])
            for k in np.where(pending == BUY)[0]:
                if shares[k] > 0 or (shares > 0).sum() >= cfg.max_positions:
                    continue
                px = P.O[d, k] * (1 + slip)
                tgt = min(cfg.max_position_pct * eq_open, cfg.risk_per_trade * eq_open / max(pstop[k], 1e-4))
                qty = math.floor(min(tgt, cash / (1 + fee)) / px)
                if qty < 1:
                    continue
                notional = qty * px; f = notional * fee
                cash -= notional + f; shares[k] = qty; full[k] = qty; avg[k] = px; ebar[k] = d
                cost[k] = notional + f; proceeds[k] = 0.0; sig[k] = d - 1; traded += notional; n_orders += 1
        pending[:] = 0
        val = (shares * P.C[d]).sum()
        eq.append(cash + val); expo.append(val / (cash + val))
        held = shares > 0
        pos_hist[d, held, 0] = shares[held] / full[held]
        pos_hist[d, held, 1] = P.C[d, held] / avg[held] - 1.0
        pos_hist[d, held, 2] = np.minimum((d - ebar[held] + 1) / cfg.max_hold, 3.0)
        if d < i1:
            a = np.asarray(policy_fn(d, pos_hist))
            for k in range(K):
                if a[k] == BUY and not held[k]:
                    pending[k] = BUY; pstop[k] = cfg.stop_atr * P.atr[d, k] / P.C[d, k]
                elif a[k] in (SELL, EXIT) and held[k]:
                    pending[k] = a[k]
    for k in np.where(shares > 0)[0]:                                      # mark open positions (stats only)
        record(k, i1, i1, proceeds[k] + shares[k] * P.C[i1, k], open_end=True)
    eqs = pd.Series(eq, index=P.dates[i0:i1 + 1])
    return dict(equity=eqs, trades=pd.DataFrame(trades), traded=traded, n_orders=n_orders, exposure=float(np.mean(expo)))

def buy_hold_curve(O, C, i0, i1, cfg):
    O, C = (np.atleast_2d(x.T).T if x.ndim == 1 else x for x in (O, C))
    n = O.shape[1]; fee, slip = cfg.fee_bps / 1e4, cfg.slippage_bps / 1e4
    sh = (cfg.initial_capital / n) / (O[i0 + 1] * (1 + slip) * (1 + fee))
    eq = np.full(i1 - i0 + 1, cfg.initial_capital, dtype=float)
    for j, d in enumerate(range(i0 + 1, i1 + 1)):
        eq[j + 1] = (sh * C[d]).sum()
    return dict(equity=eq, traded=cfg.initial_capital, n_orders=n)

def perf_metrics(eq, traded, trades=None, n_orders=None, exposure=None):
    eq = pd.Series(eq).astype(float); r = eq.pct_change().dropna()
    cum = eq.iloc[-1] / eq.iloc[0] - 1; yrs = len(r) / 252
    sd = r.std(ddof=1)
    out = dict(cumulative_return=float(cum),
               annualized_return=float((1 + cum) ** (1 / yrs) - 1) if yrs > 0 and cum > -1 else float("nan"),
               annualized_vol=float(sd * np.sqrt(252)),
               sharpe=float(r.mean() / sd * np.sqrt(252)) if sd > 0 else float("nan"),
               max_drawdown=float((eq / eq.cummax() - 1).min()),
               turnover=float(traded / eq.mean()), n_orders=int(n_orders or 0))
    if trades is not None:
        cl = trades[~trades.open_at_end] if len(trades) else trades
        out.update(trade_count=int(len(cl)), open_at_end=int(len(trades) - len(cl)),
                   win_rate=float((cl.ret > 0).mean()) if len(cl) else float("nan"),
                   avg_trade_return=float(cl.ret.mean()) if len(cl) else float("nan"),
                   avg_hold_bars=float(cl.hold_bars.mean()) if len(cl) else float("nan"))
    if exposure is not None:
        out["avg_exposure"] = exposure
    return out

def category_stats(trades):
    if trades is None or len(trades) == 0:
        return {}
    res = {}
    for cat, g in trades.groupby("category"):
        res[cat] = dict(n=int(len(g)), mean_return=float(g.ret.mean()), win_rate=float((g.ret > 0).mean()),
                        total_pnl=float(g.pnl.sum()), worst_trade=float(g.ret.min()))
    return res

## 13 · Choose the BUY-probability threshold on **validation only**, then freeze
The validation backtest is run for each threshold in `CFG.buy_thresholds`. The one with the best validation Sharpe (with at least `min_val_trades` closed trades) is frozen. The **test period is not touched** until the next cell. Note that validation data was also used for early stopping, so validation results are optimistic.

In [ ]:
i0v, i1v = int(np.where(SPLIT_IDX == 1)[0][0]), int(np.where(SPLIT_IDX == 1)[0][-1])
i0t, i1t = int(np.where(SPLIT_IDX == 2)[0][0]), int(np.where(SPLIT_IDX == 2)[0][-1])
rows = []
for thr in CFG.buy_thresholds:
    POLICY.buy_threshold = thr
    res = backtest(PANEL, make_model_policy(POLICY, PANEL, CFG), i0v, i1v, CFG)
    m = perf_metrics(res["equity"], res["traded"], res["trades"], res["n_orders"], res["exposure"])
    rows.append(dict(buy_threshold=thr, **{k: m[k] for k in ["cumulative_return", "sharpe", "max_drawdown", "trade_count"]}))
VAL_SWEEP = pd.DataFrame(rows); print(VAL_SWEEP.round(4).to_string(index=False))
ok = VAL_SWEEP[(VAL_SWEEP.trade_count >= CFG.min_val_trades) & VAL_SWEEP.sharpe.notna()]
BEST_THR = float(ok.sort_values("sharpe", ascending=False).iloc[0].buy_threshold) if len(ok) else 0.0
POLICY.buy_threshold = BEST_THR
print(f"\nFrozen buy_threshold = {BEST_THR} (selected on validation; {'fallback to argmax: too few val trades' if len(ok) == 0 else 'best val Sharpe'})")

## 14 · Held-out test evaluation
### 14.1 Classification, confusion matrix, breakout vs non-breakout conditions

In [ ]:
pte, cte = predict_proba(MODEL, Xte, DEVICE)
pred_te, cm_te, rep_te = cls_eval(yte, pte, BEST_THR)
pred_te_argmax, cm_te_am, rep_te_am = cls_eval(yte, pte, 0.0)
print(f"TEST (frozen rule: argmax + buy_threshold={BEST_THR})\n")
print(classification_report(yte, pred_te, labels=[0, 1, 2, 3], target_names=ACTION_NAMES, zero_division=0))
plot_cm(cm_te, f"TEST confusion (buy_thr={BEST_THR})")
SEG_TE = segment_report(yte, pred_te, META.seg[te_m])
seg_df = pd.DataFrame({k: {kk: vv for kk, vv in v.items() if not isinstance(vv, dict)} for k, v in SEG_TE.items()}).T
print("Performance by condition (test):"); print(seg_df.round(3).to_string())
nb = {s: int((META.seg[te_m] == s).sum()) for s in (1, 2, 3)}
if sum(nb.values()):
    print(f"\nBase rate: a naive 'buy every candidate' rule would be right on {nb[1]/sum(nb.values()):.1%} of test candidates (true={nb[1]}, false={nb[2]}, unconfirmed={nb[3]}).")
    print("Model BUY precision on candidates:", float(((pred_te == BUY) & (META.seg[te_m] == 1)).sum() / max(1, ((pred_te == BUY) & np.isin(META.seg[te_m], [1, 2, 3])).sum())))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
seg_df["accuracy"].dropna().plot.bar(ax=ax[0], title="Accuracy by condition (test)", rot=25)
if "buy_rate" in seg_df:
    seg_df.loc[["candidate_true", "candidate_false", "candidate_unconfirmed"], "buy_rate"].dropna().plot.bar(ax=ax[1], title="BUY rate on breakout candidates", rot=0)
plt.tight_layout(); plt.show()

### 14.2 Backtest: policy vs simple breakout baselines vs buy-and-hold (test period)

In [ ]:
RES = {}
RES["policy"] = backtest(PANEL, make_model_policy(POLICY, PANEL, CFG), i0t, i1t, CFG)
RES["simple_breakout"] = backtest(PANEL, make_breakout_baseline(PANEL, CFG, False), i0t, i1t, CFG)
RES["volume_confirmed_breakout"] = backtest(PANEL, make_breakout_baseline(PANEL, CFG, True), i0t, i1t, CFG)
bh = buy_hold_curve(PANEL.O, PANEL.C, i0t, i1t, CFG)
bm = buy_hold_curve(PANEL.bench_O, PANEL.bench_C, i0t, i1t, CFG)
idx_t = PANEL.dates[i0t:i1t + 1]
EQ = pd.DataFrame({"policy": RES["policy"]["equity"], "simple_breakout": RES["simple_breakout"]["equity"],
                   "volume_confirmed_breakout": RES["volume_confirmed_breakout"]["equity"],
                   "buy_hold_equal_weight": pd.Series(bh["equity"], index=idx_t),
                   "benchmark_buy_hold": pd.Series(bm["equity"], index=idx_t)})
BT = {k: perf_metrics(v["equity"], v["traded"], v["trades"], v["n_orders"], v["exposure"]) for k, v in RES.items()}
BT["buy_hold_equal_weight"] = perf_metrics(bh["equity"], bh["traded"], None, bh["n_orders"])
BT["benchmark_buy_hold"] = perf_metrics(bm["equity"], bm["traded"], None, bm["n_orders"])
bt_df = pd.DataFrame(BT).T[["cumulative_return", "annualized_return", "sharpe", "max_drawdown", "turnover", "trade_count", "win_rate", "avg_exposure"]]
print(f"TEST window {idx_t[0].date()} -> {idx_t[-1].date()} | fee {CFG.fee_bps}bp + slip {CFG.slippage_bps}bp per side | threshold {BEST_THR}")
print(bt_df.round(4).to_string())

fig, ax = plt.subplots(2, 1, figsize=(11, 7), sharex=True, gridspec_kw={"height_ratios": [3, 1.3]})
(EQ / EQ.iloc[0]).plot(ax=ax[0]); ax[0].set_title("Test-period equity (normalised). Research only, not a forecast"); ax[0].grid(alpha=.3)
(EQ / EQ.cummax() - 1).plot(ax=ax[1], legend=False); ax[1].set_title("Drawdown"); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

### 14.3 False breakouts & losses after failed breakouts
Trades are grouped by the **hindsight** outcome of the candidate they were entered on (evaluation only, never an input). `false_breakout` = the breakout failed within the horizon. `non_candidate` = entry on a bar that did not satisfy the candidate definition (e.g. baseline entries out of loose ranges).

In [ ]:
FB = {k: category_stats(RES[k]["trades"]) for k in ["policy", "simple_breakout", "volume_confirmed_breakout"]}
rows = []
for strat, cats in FB.items():
    for cat, s in cats.items():
        rows.append(dict(strategy=strat, category=cat, **s))
FB_DF = pd.DataFrame(rows)
print(FB_DF.round(4).to_string(index=False) if len(FB_DF) else "no trades")
# candidate-level avoidance on the test samples
cand_false = (META.seg[te_m] == 2)
if cand_false.sum():
    print(f"\nFalse-breakout avoidance (test candidate bars): policy did NOT buy {1 - float((pred_te[cand_false] == BUY).mean()):.1%} of {int(cand_false.sum())} false-breakout candidate bars; a naive 'buy every breakout' rule avoids 0% by construction.")
if len(FB_DF):
    pv = FB_DF.pivot_table(index="category", columns="strategy", values="mean_return")
    pv.plot.bar(figsize=(8, 3.4), title="Mean trade return by breakout outcome category (test)", rot=15); plt.axhline(0, color="k", lw=.6); plt.tight_layout(); plt.show()
tt = RES["policy"]["trades"]
if len(tt):
    tk = tt.ticker.value_counts().index[0]; k = PANEL.tickers.index(tk)
    fig, ax = plt.subplots(figsize=(11, 3.6)); ax.plot(idx_t, PANEL.C[i0t:i1t + 1, k], lw=1, label=f"{tk} close")
    for _, r in tt[tt.ticker == tk].iterrows():
        ax.axvline(r.entry_date, color="g", alpha=.5); ax.axvline(r.exit_date, color="r", alpha=.5)
    ax.set_title(f"Policy entries (green) / exits (red) on {tk}, test period"); plt.tight_layout(); plt.show()

## 15 · Export artifacts
Writes `breakout_policy.pt`, `breakout_config.json`, `feature_schema.json`, `preprocessing.pkl`, `evaluation_metrics.json`, `equity_curve.csv` (plus an extra `trades_test.csv`).

In [ ]:
def jsonable(o):
    if isinstance(o, dict): return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [jsonable(v) for v in o]
    if isinstance(o, np.ndarray): return jsonable(o.tolist())
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating, float)):
        o = float(o); return None if (math.isnan(o) or math.isinf(o)) else o
    if isinstance(o, (pd.Timestamp,)): return str(o.date())
    return o

AD = Path(CFG.artifact_dir); AD.mkdir(parents=True, exist_ok=True)
PATHS = {n: str(AD / n) for n in ["breakout_policy.pt", "breakout_config.json", "feature_schema.json",
                                   "preprocessing.pkl", "evaluation_metrics.json", "equity_curve.csv"]}

POLICY.buy_threshold = BEST_THR
POLICY.save(PATHS["breakout_policy.pt"])

DESC = {"ret_1": "1-bar log return", "ret_5": "5-bar log return", "ret_10": "10-bar log return", "ret_20": "20-bar log return",
        "rsi_14": "(RSI14-50)/50", "atr_pct": "ATR14/close", "vol_rel_20": "log(volume / mean volume of bars t-20..t-1), clipped",
        "gap": "open_t/close_{t-1}-1", "sma20_50": "SMA20/SMA50-1", "close_vs_sma50": "close/SMA50-1", "adx_14": "ADX14/100",
        "rel_strength_20": "ret_20 minus benchmark 20-bar log return",
        "pos_frac": "current shares / shares at entry (0..1)", "unrealized_pnl": "close/avg_entry-1", "holding_bars_norm": "bars held / max_hold",
        "profile_risk_tolerance": "optional trader profile (const 0.5 by default)", "profile_horizon_pref": "optional trader profile (const 0.5 by default)",
        "dist_prior_high_atr": "(close - prior_high)/ATR; prior range = bars t-N..t-1", "dist_prior_low_atr": "(close - prior_low)/ATR",
        "range_width_pct": "(prior_high-prior_low)/close", "range_width_atr": "(prior_high-prior_low)/ATR",
        "compression_ratio": "range_width_pct / its 120-bar mean (<1 = compressed)", "atr_expansion": "log(mean TR 5 / mean TR 30)",
        "true_range_atr": "true range / ATR14", "vol_rel_3": "log(mean volume 3 / mean volume t-20..t-1)",
        "close_location": "position of close in today's range, -1..1", "breakout_up": "close > prior_high (raw flag)",
        "breakdown": "close < prior_low", "breakout_active": "a fresh breakout occurred within the last K bars",
        "bars_since_breakout_norm": "bars since fresh breakout / K (1 if inactive)", "followthrough_atr": "(close - frozen breakout level)/ATR while active",
        "back_inside_range": "active and close < frozen breakout level"}
group_of = lambda c: "base_market" if c in BASE_MARKET else "base_position" if c in BASE_POSITION else "base_profile" if c in BASE_PROFILE else "extension_breakout"
schema = dict(
    policy=BreakoutPolicy.NAME, version=BreakoutPolicy.VERSION, seq_len=CFG.seq_len, n_features=F_DIM,
    input_shape=["batch", CFG.seq_len, F_DIM], input_is_raw_unscaled=True,
    columns=[dict(index=i, name=c, group=group_of(c), scaling=("fit_on_train" if i in MKT_IDX else "fixed"), description=DESC[c]) for i, c in enumerate(FEATURE_COLUMNS)],
    base_block=dict(indices=[0, D_BASE - 1], columns=BASE, note="ASSUMED shared block. Align with other SP policies"),
    extension_block=dict(indices=[D_BASE, F_DIM - 1], columns=EXT, adapter="BreakoutPolicy.build_state(base, extra=None)"),
    action_map={str(k): v for k, v in ACTIONS.items()},
    action_semantics=dict(HOLD="no change", BUY="open long if flat", SELL=f"reduce long by {CFG.sell_fraction:.0%}", EXIT="close all"),
    timing=dict(decision="after close of bar t", execution="open of bar t+1",
                range_window="bars t-N..t-1 (current bar excluded)", volume_avg_window="bars t-20..t-1",
                labels_use_future_bars="t+1..t+H (labels only, never inputs)"),
    outputs=dict(probabilities="[HOLD,BUY,SELL,EXIT]", action="int, buy_threshold applied", confidence="learned P(argmax correct) in [0,1]"))
Path(PATHS["feature_schema.json"]).write_text(json.dumps(jsonable(schema), indent=2))

config_out = dict(policy=BreakoutPolicy.NAME, version=BreakoutPolicy.VERSION, artifact="breakout_policy.pt",
                  model=MODEL.cfg, seq_len=CFG.seq_len, decision=dict(buy_threshold=BEST_THR),
                  config=asdict(CFG), class_weights=dict(zip(ACTION_NAMES, CLASS_W)), device_trained=str(DEVICE),
                  environment=dict(python=sys.version.split()[0], torch=str(torch.__version__), numpy=np.__version__, pandas=pd.__version__),
                  label_type="rule-derived proxy labels (NOT human demonstrations); entry labels use forward outcome over horizon H")
Path(PATHS["breakout_config.json"]).write_text(json.dumps(jsonable(config_out), indent=2))

pre_pkl = dict(mean=PREPROC["mean"], std=PREPROC["std"], clip=PREPROC["clip"], columns=FEATURE_COLUMNS,
               ext_neutral_raw=PREPROC["ext_neutral_raw"], fitted_on=PREPROC["fitted_on"], format="plain dict (no custom classes)")
with open(PATHS["preprocessing.pkl"], "wb") as fh:
    pickle.dump(pre_pkl, fh)

metrics = dict(
    disclaimer="Research / paper trading only. Historical backtest results are not evidence of future profitability.",
    data_source=CFG.data_source, tickers=PANEL.tickers, benchmark=CFG.benchmark,
    periods=dict(train=f"{PANEL.dates[SPLIT_IDX == 0][0].date()}..{CFG.train_end}", val=f"..{CFG.val_end}",
                 test=f"{idx_t[0].date()}..{idx_t[-1].date()}"),
    label_counts={sn: {ACTIONS[a]: int((Y[META.split == i] == a).sum()) for a in range(4)} for i, sn in enumerate(SPLIT_NAMES)},
    buy_threshold=BEST_THR, validation_threshold_sweep=VAL_SWEEP.to_dict("records"),
    classification_test=dict(report=rep_te, confusion_matrix=cm_te.tolist(), labels=ACTION_NAMES, argmax_only=dict(report=rep_te_am, confusion_matrix=cm_te_am.tolist())),
    classification_val_argmax=dict(report=rep_va, confusion_matrix=cm_va.tolist()),
    by_condition_test=SEG_TE, backtest_test=BT,
    false_breakout_analysis=FB, backtest_assumptions=dict(fee_bps=CFG.fee_bps, slippage_bps=CFG.slippage_bps, max_positions=CFG.max_positions,
        max_position_pct=CFG.max_position_pct, risk_per_trade=CFG.risk_per_trade, initial_capital=CFG.initial_capital, execution="next-bar open"),
    seed=CFG.seed, torch_version=str(torch.__version__))
Path(PATHS["evaluation_metrics.json"]).write_text(json.dumps(jsonable(metrics), indent=2))

eq_out = EQ.copy(); eq_out.index.name = "date"; eq_out["policy_drawdown"] = eq_out["policy"] / eq_out["policy"].cummax() - 1
eq_out.to_csv(PATHS["equity_curve.csv"])
RES["policy"]["trades"].to_csv(AD / "trades_test.csv", index=False)
for n, p in PATHS.items():
    print(f"{n:26s} {os.path.getsize(p):>10,d} bytes  {p}")

## 16 · Load & inference check (deterministic, no downloads)
Reload the artifact from disk, confirm identical outputs across loads, equality with the in-memory model, and run the documented adapter path.

In [ ]:
reloaded_a = BreakoutPolicy.load(PATHS["breakout_policy.pt"], device="cpu")
reloaded_b = BreakoutPolicy.load(PATHS["breakout_policy.pt"], device="cpu")
sample_idx = int(np.where(META.seg[te_m] == 1)[0][0]) if (META.seg[te_m] == 1).any() else 0
state = X_RAW[te_m][sample_idx]                                  # raw (seq_len, F) test window
oa, ob = reloaded_a.predict(state), reloaded_b.predict(state)
assert oa["probabilities"] == ob["probabilities"], "loads are not deterministic"
POLICY.net.cpu(); POLICY.device = torch.device("cpu"); inmem = POLICY.predict(state)
assert np.allclose(oa["probabilities"], inmem["probabilities"], atol=1e-5), "reloaded != in-memory"
print("deterministic load OK | reloaded == in-memory OK\n")
print(json.dumps(oa, indent=2))
# adapter path: base block only -> extension features neutral
o_ad = reloaded_a.predict({"base": state[:, :D_BASE]})
print("\nadapter w/o extension features ->", o_ad["action_name"], "| warnings:", o_ad["warnings"])
try:
    reloaded_a.predict(state[:, :-1])
except ValueError as e:
    print("shape guard OK:", e)

## 17 · Package for download

In [ ]:
zip_path = shutil.make_archive("/content/sp04_breakout_artifacts", "zip", CFG.artifact_dir)
print("zip:", zip_path)
try:
    from google.colab import files
    for p in list(PATHS.values()) + [zip_path]:
        files.download(p)
except ImportError:
    print("(not in Colab: files are on disk at the paths above)")

## 18 · Final summary

### Exact artifact paths (default `CFG.artifact_dir`)
```
/content/sp04_breakout_artifacts/breakout_policy.pt
/content/sp04_breakout_artifacts/breakout_config.json
/content/sp04_breakout_artifacts/feature_schema.json
/content/sp04_breakout_artifacts/preprocessing.pkl
/content/sp04_breakout_artifacts/evaluation_metrics.json
/content/sp04_breakout_artifacts/equity_curve.csv
/content/sp04_breakout_artifacts/trades_test.csv        (extra)
/content/sp04_breakout_artifacts.zip                    (all of the above)
```

### Assumptions
1. **Proxy labels, not human demonstrations.** Entry labels are hindsight outcomes (follow-through vs failure within `H` bars). Management labels come from causal stop/trail/failure/target rules applied to *shadow positions* entered at every candidate.
2. **Shared base block (17 cols) is assumed.** I did not have the other policies' exact input spec. Align `BASE_*` lists if they differ.
3. **Action semantics:** BUY = open long, SELL = reduce by `sell_fraction`, EXIT = close all. Long-only. Confirm the Meta-RL layer uses the same meaning of SELL.
4. Adjusted prices from yfinance, daily bars, signals at the close, fills at the next open.
5. The `buy_threshold` and early stopping both use the validation period. The test period is evaluated once after freezing.
6. Profile/risk features are constants unless you wire real values in.

### Known limitations
* **Label noise / non-identifiability:** the model cannot see the future, so many true and false breakouts are statistically indistinguishable at decision time. Expect modest classification skill. Interpret the BUY-precision vs base-rate comparison honestly.
* **Rule-derived management labels** teach the model the proxy rules, not an optimal exit policy. In-position rows come from shadow positions (position state is simulated, not observed from a real account).
* **Survivorship and selection bias:** the default tickers are today's large caps. Delisted or failed names are absent. Past leadership flatters breakout strategies.
* **Regime dependence:** one train/val/test split. No walk-forward or multi-seed variance estimate. Results can change materially with the seed, thresholds and period.
* **Execution realism:** flat bps costs, no market impact, no gaps through stops beyond the open fill, no borrow/short, whole shares, slots filled in ticker order, daily bars only.
* Multiple-testing risk: hyper-parameters (`range_max_width_atr`, `horizon`, `up_atr`, …) are tunable. Re-tuning on test results invalidates the test.
* Bitwise reproducibility holds only for the same hardware / library versions.

### Compatibility checklist for the Meta-RL model
- [ ] Action ids `0=HOLD, 1=BUY, 2=SELL, 3=EXIT` identical across all SP policies
- [ ] SELL = partial reduction, EXIT = full close, and BUY while long is a no-op (SP-04 definitions) agree with the Meta-RL environment
- [ ] `seq_len=32` and `F=32` (`feature_schema.json`). Meta-RL passes **raw, unscaled** features. The `.pt` applies scaling internally
- [ ] First 17 columns match the shared base block of the other policies. Otherwise edit `BASE_*` and retrain
- [ ] Extension features supplied via `build_state(base, extra)`, or neutral-filled (warning returned)
- [ ] Position columns (`pos_frac`, `unrealized_pnl`, `holding_bars_norm`) are computed by the caller with the same definitions as in `feature_schema.json`
- [ ] Output contract consumed: `probabilities` (list of 4, action order), `action`, `confidence` (learned correctness score, not a calibrated profit probability), `max_prob`
- [ ] `buy_threshold` (in `.pt`) is already applied inside `action`; use `raw_argmax` / `probabilities` if Meta-RL wants its own gating
- [ ] Loading: `BreakoutPolicy.load(path, device)`, `weights_only=True`, no network/yfinance/sklearn needed
- [ ] Inputs must be finite. NaN/inf raise `ValueError`. Short history is left-padded with a warning
- [ ] Next-bar execution convention: decision on close of `t`, fill at open of `t+1`
- [ ] `evaluation_metrics.json` has `data_source="yfinance"` (not `synthetic`) before any downstream use
- [ ] Paper trading / research only. No live-capital use without independent validation